In [2]:
from langgraph.checkpoint.memory import MemorySaver
import operator
import os


from dotenv import load_dotenv
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import Literal, TypedDict, Annotated, List, Optional
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END, add_messages
from langchain.tools import tool
from langgraph.types import Command, interrupt

load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()

documents = [
    "LangGraph is a library for building stateful multi-agent applications using graphs.",
    "RAG stands for Retrieval Augmented Generation. It combines retrieval with LLM generation.",
    "The Eiffel Tower is located in Paris, France and was built in 1889.",
    "Vector databases store embeddings for semantic similarity search.",
    "Python is a high-level programming language known for its simplicity.",
    "CRAG improves RAG by grading retrieved documents and rewriting queries when retrieval fails.",
]
grade_system_prompt="You are answer evaluator of question.You checked the answer provided by the process is relevant or not. If relevent set the value of is_relevant to True"

system_prompt = """You are a supervisor managing a multiple agent for various task.
Follow this exact workflow based on user request :
- If user requested for any calculation then route to calculator_agent
- If user requested for any rag related question route to faq_agent
- If user want to speak to customer care route to escalator_agen
- For other scenario: route to FINISH (both done)
"""

class CapStoneState(TypedDict):
    messages:Annotated[List, add_messages]
    next:  str
    documents:     List[str]
    generation:    str
    rewrite_count: int
    action_details: str
    status:        Optional[Literal["pending", "approved", "rejected"]]

class RouteDecision(BaseModel):
    next: Literal["faq_agent", "calculator_agent", "escalate_agent", "FINISH"]

def supervisor_node(state:CapStoneState):
    msg =state["messages"][-1].content
    struc_res = llm_openai.with_structured_output(RouteDecision)
    response = struc_res.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"User request: {msg}")
    ])
    return {"next": response.next}

class FaQState(CapStoneState):
    pass

class GradeOutput(BaseModel):
    is_relevant: bool

def retrieve_node(state: FaQState):
    msg =state["messages"][-1].content
    question = msg.lower()
    # keep any doc that shares a word with the question
    relevant = [d for d in documents if any(w in d.lower() for w in question.split())]
    return {"documents": relevant if relevant else documents[:2]}  # fallback

def grade_node(state: FaQState):
     # for each doc, ask LLM: does this doc help answer the question?
    # collect only relevant ones → {"documents": relevant_docs}
    # if none relevant → {"documents": []}
    msg =state["messages"][-1].content
    question = msg.lower()
    struc_res = llm_openai.with_structured_output(GradeOutput)
    relevant_docs = []
    for doc in state["documents"]:
        response = struc_res.invoke([
            SystemMessage(content=grade_system_prompt),
            HumanMessage(content=f"Question: {question}\nDocument: {doc}")
        ])
        if response.is_relevant:
            relevant_docs.append(doc)

    return {"documents": relevant_docs}

def rewrite_node (state: FaQState):
    # prompt: "Rewrite this question to improve document retrieval. Make it more specific."
    # returns {"question": rewritten, "rewrite_count": state["rewrite_count"] + 1}
    msg =state["messages"][-1].content
    question = msg.lower()
    response = llm_openai.invoke([
        SystemMessage(content="Rewrite this question to improve document retrieval. Make it more specific and add relevant keywords."),
        HumanMessage(content=question)
    ])
    return {"messages": [response], "rewrite_count": state["rewrite_count"] + 1}

def generate_node (state: FaQState):
    # prompt: "Answer the question using these documents: {docs}"
    docs_text = '\n\n'.join(state["documents"])
    msg =state["messages"][-1].content
    question = msg.lower()
    response = llm_openai.invoke([
        SystemMessage(content=f"Answer the question using only these documents:\n\n{docs_text}"),
        HumanMessage(content=question)
    ])
    return {"generation": response.content}

def route_after_grade(state: FaQState) -> str:
    if len(state["documents"]) > 0:
        return "generate"
    if state["rewrite_count"] >= 2:
        return "generate"   # safety — generate even with no docs after 2 rewrites
    return "rewrite"

g_graph=StateGraph(FaQState)
g_graph.add_node('retrieve_node',retrieve_node)
g_graph.add_node('grade_node',grade_node)
g_graph.add_node('rewrite_node',rewrite_node)
g_graph.add_node('generate_node',generate_node)
g_graph.add_edge(START,'retrieve_node')
g_graph.add_edge('retrieve_node','grade_node')
g_graph.add_conditional_edges('grade_node',route_after_grade,{"generate":'generate_node','rewrite':'rewrite_node'})
g_graph.add_edge('rewrite_node','retrieve_node')
g_graph.add_edge('generate_node',END)

faq_app=g_graph.compile()

@tool
def multiply(a:int,b:int)->int:
   """
    This tool is used for multiplication of two numbers.
   """
   return a*b

'''
add: used for addition of 2 number
'''
@tool
def add(a:int,b:int)->int:
    """
    This tool is used for addition of two numbers.
   """
    return a+b

llm_with_tools=llm_openai.bind_tools([multiply,add])

class CalcAgentState(CapStoneState):
    pass

def calc_agent_node(state:CapStoneState):
    response = llm_with_tools.invoke(state["messages"])   # pass ALL messages
    return {"messages": [response]}

def tool_node(state:CapStoneState):
    response = state["messages"][-1]
    tools_map = {"multiply": multiply, "add": add}
    messages=[]
    for tool_call in response.tool_calls:
        actual_tool = tools_map[tool_call["name"]]
        result = actual_tool.invoke(tool_call["args"])
        messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))
    return {"messages":messages}

def conditional_node(state:CapStoneState):
    response = state["messages"][-1]
    if response.tool_calls:
        return "tool_node"
    return "end"

f_graph=StateGraph(CalcAgentState)
f_graph.add_node("calc_agent_node",calc_agent_node)
f_graph.add_node("tool_node",tool_node)
f_graph.add_edge(START,"calc_agent_node")
f_graph.add_conditional_edges("calc_agent_node",conditional_node,{"tool_node":"tool_node","end":END})
f_graph.add_edge("tool_node","calc_agent_node")

calc_app=f_graph.compile()

class ApprovalState(CapStoneState):
    pass

def approval_node(state: ApprovalState) -> Command[Literal["proceed", "cancel"]]:
    # Expose details so the caller can render them in a UI
    decision = interrupt({
        "question": "Approve this action?",
        "details": state["action_details"],
    })

    # Route to the appropriate node after resume
    return Command(goto="proceed" if decision else "cancel")


def proceed_node(state: ApprovalState):
    return {"status": "approved"}


def cancel_node(state: ApprovalState):
    return {"status": "rejected"}

builder = StateGraph(ApprovalState)
builder.add_node("approval", approval_node)
builder.add_node("proceed", proceed_node)
builder.add_node("cancel", cancel_node)
builder.add_edge(START, "approval")
builder.add_edge("proceed", END)
builder.add_edge("cancel", END)
memory = MemorySaver()
exception_agent = builder.compile(checkpointer=memory)

def route_supervisor(state: CapStoneState) -> str:
    return state["next"]

super_graph=StateGraph(CapStoneState)
super_graph.add_node('supervisor_node',supervisor_node)
super_graph.add_node('faq_app',faq_app)
super_graph.add_node('calc_app',calc_app)
super_graph.add_node('exception_agent',exception_agent)
super_graph.add_edge(START,'supervisor_node')
super_graph.add_conditional_edges('supervisor_node', route_supervisor, {
    "faq_agent":        "faq_app",
    "calculator_agent": "calc_app",
    "escalate_agent":   "exception_agent",
    "FINISH":           END
})
app = super_graph.compile(checkpointer=MemorySaver())
config1 = {"configurable": {"thread_id": "test-faq"}}
config2 = {"configurable": {"thread_id": "test-calc"}}
config3 = {"configurable": {"thread_id": "test-escalate"}}

base_state = {"next": "", "documents": [], "generation": "",
              "rewrite_count": 0, "action_details": "", "status": "pending"}

# q1 — FAQ
r1 = app.invoke({**base_state, "messages": [HumanMessage(content="What is RAG?")]}, config=config1)
print("FAQ:", r1.get("generation", "")[:100])

# q2 — Calculator
r2 = app.invoke({**base_state, "messages": [HumanMessage(content="What is (23 + 17) * 4?")]}, config=config2)
print("CALC:", r2["messages"][-1].content)

# q3 — Escalate (interrupt)
app.invoke({**base_state,
            "messages": [HumanMessage(content="I want to speak to someone urgently")],
            "action_details": "User requested human escalation"}, config=config3)
state = app.get_state(config3)
print("INTERRUPT pending:", state.next)
app.invoke(None, config=config3)   # approve
print("Status:", app.get_state(config3).values.get("status"))






C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RouteDecision(next='faq_agent'), input_type=RouteDecision])
  return self.__pydantic_serializer__.to_python(
C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=GradeOutput(is_relevant=False), input_type=GradeOutput])
  return self.__pydantic_serializer__.to_python(
C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=GradeOutput

FAQ: Retrieval‑augmented generation (RAG) is a technique that augments a language model’s generation with


C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RouteDecision(next='calculator_agent'), input_type=RouteDecision])
  return self.__pydantic_serializer__.to_python(


CALC: The result is **160**.
INTERRUPT pending: ('exception_agent',)
Status: pending


C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RouteDecision(next='escalate_agent'), input_type=RouteDecision])
  return self.__pydantic_serializer__.to_python(
